In [ ]:
%pip install -q "datasets>=3.0,<5" "bert-score>=0.3.13,<0.4" tqdm ijson

In [ ]:
import os, json, math, random, hashlib, zipfile, gc, warnings, sys, time
from pathlib import Path
from collections import Counter
from getpass import getpass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tqdm.auto import tqdm
from datasets import load_dataset

RUN_SMOKE_TEST = True
RUN_TRAINING = True
RUN_LOCAL_GENERATION_EVAL = True
RUN_OFFICIAL_GENERATION = True
RUN_DECODING_SWEEP = True
LOCAL_EVAL_EXAMPLES = 64
MAX_TEST_EXAMPLES = None
OUTPUT_WORDS = 128
TEMPERATURE = 0.8
TOP_K = 50
TOP_P = 0.90
GENERATION_TOKEN_BUDGETS = (1024, 2048)
MAX_GENERATION_TOKENS = max(GENERATION_TOKEN_BUDGETS)
MAX_BPE_TOKENS_PER_WORD = 24
DECODING_TEMPERATURES = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
EVAL_MEM_LEN_OPTIONS = (512, 1024)
DATASET_ID = "Exploration-Lab/CS-781-Capstone"
DATASET_CONFIG = "phase_1"
SEED = 42
MAX_TRAIN_DOCS = 200_000
MAX_CORPUS_CHARS = 200_000_000
MAX_CHARS_PER_DOC = 50_000
HF_FS_BLOCK_SIZE = 1 * 1024 * 1024
SOURCE_FILES = {
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}
MAX_CACHED_WORD_ENCODINGS = 50_000
MAX_TOKENIZER_DOCS = 64
TARGET_VOCAB_SIZE = 10_000
TRAIN_TOKEN_BUDGET = 3_000_000
VALID_TOKEN_BUDGET = 200_000
VALID_FRACTION = 0.02
SEGMENT_LEN = 256
TRAIN_MEM_LEN = 512
EVAL_MEM_LEN = 1024
MICRO_BATCH_SIZE = 8
EPOCHS = 6
GRAD_ACCUM = 2
EVAL_EVERY_STEPS = 250
MAX_EVAL_BATCHES = 100
WARMUP_FRACTION = 0.05
LEARNING_RATE = 3e-4
WARM_START_LEARNING_RATE = 1e-5
CONTINUE_FROM_EXISTING_CHECKPOINT = False
WEIGHT_DECAY = 0.10
GRAD_CLIP = 1.0
USE_AMP = True
USE_GRAD_CHECKPOINTING = False
ARCHITECTURE_NAME = "transformer_xl_relative_recurrence_tuned"
MODEL_D = 768
MODEL_HEADS = 12
MODEL_LAYERS = 12
MODEL_FF = 3072
MODEL_DROPOUT = 0.10
RELATIVE_POSITION_BASE = 10_000.0
OUTPUT_DIR = Path("phase1_transformer_xl_250m_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_PATH = OUTPUT_DIR / "legal_bpe_tokenizer.json"
LATEST_CHECKPOINT_PATH = OUTPUT_DIR / "latest_checkpoint.pt"
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "best_checkpoint.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
elif RUN_TRAINING:
    raise RuntimeError("Enable the Kaggle GPU accelerator before running this notebook.")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face read token: ")
if not HF_TOKEN:
    raise RuntimeError("A Hugging Face token is required to access the gated Phase-1 dataset.")

def stable_json_hash(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")).hexdigest()

def _json_safe(value):
    if value is None or isinstance(value, (str, bool)):
        return value
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        number = float(value)
        return number if math.isfinite(number) else None
    if isinstance(value, (int, np.integer)):
        return int(value)
    if isinstance(value, Path):
        return str(value)
    return value

def save_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(_json_safe(obj), f, ensure_ascii=False, indent=2, allow_nan=False)

def make_grad_scaler(enabled):
    if not enabled: return None
    try: return torch.amp.GradScaler("cuda", enabled=True)
    except (AttributeError, TypeError): return torch.cuda.amp.GradScaler(enabled=True)
print("Train-token budget per epoch:", f"{TRAIN_TOKEN_BUDGET:,}")
print("Schedule:", EPOCHS, "epochs; segment", SEGMENT_LEN, "memory", TRAIN_MEM_LEN)

In [ ]:
from huggingface_hub import HfFileSystem
import ijson

hffs = HfFileSystem(token=HF_TOKEN, block_size=HF_FS_BLOCK_SIZE)

def iter_source_records(source_path):
    remote_path = f"datasets/{DATASET_ID}/{source_path}"
    with hffs.open(remote_path, "rb") as fh:

        yield from ijson.items(fh, "item")

def normalize_train_text(record):
    if not isinstance(record, dict) or "text" not in record:
        raise RuntimeError(
            "Unexpected Phase-1 record: every training record must contain a 'text' field."
        )
    text = record["text"]
    if not isinstance(text, str):
        return ""
    return text.strip()

def is_validation_document(source_name, source_row_index):
    key = f"{SEED}|{source_name}|{source_row_index}".encode("utf-8")
    digest = hashlib.blake2b(key, digest_size=8).digest()
    bucket = int.from_bytes(digest, byteorder="little", signed=False) % 10_000
    threshold = int(round(VALID_FRACTION * 10_000))
    return bucket < threshold

def iter_bounded_source_documents(source_name):
    source_path = SOURCE_FILES[source_name]
    source_char_budget = max(1, MAX_CORPUS_CHARS // len(SOURCE_FILES))
    source_doc_budget = max(1, MAX_TRAIN_DOCS // len(SOURCE_FILES))

    char_count = 0
    doc_count = 0
    truncated_docs = 0

    for row_index, record in enumerate(iter_source_records(source_path)):
        if doc_count >= source_doc_budget or char_count >= source_char_budget:
            break

        text = normalize_train_text(record)
        if not text:
            continue

        if len(text) > MAX_CHARS_PER_DOC:
            text = text[:MAX_CHARS_PER_DOC]
            truncated_docs += 1

        remaining_chars = source_char_budget - char_count
        if len(text) > remaining_chars:
            text = text[:remaining_chars]

        if not text:
            break

        is_valid = is_validation_document(source_name, row_index)
        doc_count += 1
        char_count += len(text)
        yield source_name, row_index, text, is_valid

        if len(text) >= remaining_chars:
            break

    return

load_audit = {
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "split": "train",
    "seed": SEED,
    "loader": "HfFileSystem + ijson incremental JSON parsing",
    "source_files": SOURCE_FILES,
    "source_char_budget": int(MAX_CORPUS_CHARS // len(SOURCE_FILES)),
    "source_doc_budget": int(MAX_TRAIN_DOCS // len(SOURCE_FILES)),
    "max_chars_per_doc": MAX_CHARS_PER_DOC,
    "validation_fraction": VALID_FRACTION,
}

print("Phase-1 training loader: remote JSON is parsed incrementally; no 40k-document RAM list is created.")
print("Per-source document budget:", MAX_TRAIN_DOCS // len(SOURCE_FILES))
print("Per-source character budget:", MAX_CORPUS_CHARS // len(SOURCE_FILES))

for source_name in SOURCE_FILES:
    count = 0
    chars = 0
    valid = 0
    truncated = 0
    for _, row_index, text, is_valid in iter_bounded_source_documents(source_name):
        count += 1
        chars += len(text)
        valid += int(is_valid)
        if len(text) == MAX_CHARS_PER_DOC:
            truncated += 1
        if count >= 3:

            break
    print(f"{source_name}: first 3 parsed records OK | sample_chars={chars:,} | sample_valid={valid}")

save_json(OUTPUT_DIR / "training_document_load_metadata.json", load_audit)

In [ ]:
split_metadata = {
    "seed": SEED,
    "validation_fraction": VALID_FRACTION,
    "split_unit": "document",
    "storage": "streamed; not materialized in RAM",
    "source_files": SOURCE_FILES,
    "source_doc_budget_each": MAX_TRAIN_DOCS // len(SOURCE_FILES),
    "source_char_budget_each": MAX_CORPUS_CHARS // len(SOURCE_FILES),
}
save_json(OUTPUT_DIR / "data_split_metadata.json", split_metadata)
print(split_metadata)

In [ ]:
class ByteBPE:
    def __init__(self, vocab_size=5000):
        self.target_vocab_size = vocab_size
        self.specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.eow = "</w>"
        self.merges = []
        self.itos = []
        self.stoi = {}
        self.payload = {}
        self._encode_cache = {}

    def _init_symbols(self):
        self.itos = self.specials + [f"<0x{i:02X}>" for i in range(256)] + [self.eow]
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.payload = {i: bytes([i - 4]) for i in range(4, 260)}
        self.payload[260] = b""

    @staticmethod
    def _words(texts):
        counts = Counter()
        for text in texts:
            for word in str(text).split():
                counts[word] += 1
        return counts

    def fit(self, texts, vocab_size=None):
        if vocab_size is not None:
            self.target_vocab_size = vocab_size
        self._init_symbols()
        self._encode_cache = {}
        counts = self._words(texts)
        seqs = {
            w: [b + 4 for b in w.encode("utf-8")] + [260]
            for w in counts
        }
        freq = dict(counts)

        while len(self.itos) < self.target_vocab_size:
            pair_counts = Counter()
            for w, seq in seqs.items():
                for pair in zip(seq, seq[1:]):
                    pair_counts[pair] += freq[w]

            if not pair_counts:
                break

            best_count = max(pair_counts.values())
            if best_count < 2:
                break

            pair = min((p for p, c in pair_counts.items() if c == best_count))

            new_id = len(self.itos)
            self.merges.append(pair)
            self.itos.append(f"<{pair[0]}+{pair[1]}>")
            self.stoi[self.itos[-1]] = new_id
            self.payload[new_id] = (
                self.payload.get(pair[0], b"")
                + self.payload.get(pair[1], b"")
            )

            for w, seq in seqs.items():
                out = []
                j = 0
                while j < len(seq):
                    if j + 1 < len(seq) and (seq[j], seq[j + 1]) == pair:
                        out.append(new_id)
                        j += 2
                    else:
                        out.append(seq[j])
                        j += 1
                seqs[w] = out

        self.vocab_size = len(self.itos)
        return self

    def encode_word(self, word):
        word = str(word)
        cached = self._encode_cache.get(word)
        if cached is not None:
            return list(cached)

        seq = [b + 4 for b in word.encode("utf-8")] + [260]
        for rank, pair in enumerate(self.merges):
            out = []
            j = 0
            while j < len(seq):
                if j + 1 < len(seq) and (seq[j], seq[j + 1]) == pair:
                    out.append(261 + rank)
                    j += 2
                else:
                    out.append(seq[j])
                    j += 1
            seq = out

        if len(self._encode_cache) < MAX_CACHED_WORD_ENCODINGS:
            self._encode_cache[word] = tuple(seq)
        return seq

    def encode(self, text):
        return [i for word in str(text).split() for i in self.encode_word(word)]

    def to_json(self):
        return {
            "merges": [list(p) for p in self.merges],
            "itos": self.itos,
            "vocab_size": self.vocab_size,
        }

    @classmethod
    def from_json(cls, payload):
        tokenizer = cls(vocab_size=int(payload["vocab_size"]))
        tokenizer._init_symbols()
        tokenizer.merges = [tuple(map(int, pair)) for pair in payload["merges"]]
        for pair in tokenizer.merges:
            new_id = len(tokenizer.itos)
            symbol = f"<{pair[0]}+{pair[1]}>"
            tokenizer.itos.append(symbol)
            tokenizer.stoi[symbol] = new_id
            tokenizer.payload[new_id] = (
                tokenizer.payload.get(pair[0], b"")
                + tokenizer.payload.get(pair[1], b"")
            )
        tokenizer.vocab_size = len(tokenizer.itos)
        tokenizer._encode_cache = {}
        if tokenizer.itos != payload.get("itos"):
            raise ValueError("Saved ByteBPE vocabulary and merge list are inconsistent.")
        return tokenizer

def build_word_end_flags(bpe):

    ends_word = [False] * len(bpe.itos)
    ends_word[260] = True
    for rank, (left, right) in enumerate(bpe.merges):
        new_id = 261 + rank
        ends_word[new_id] = ends_word[left] or ends_word[right]
    return ends_word

def decode_bpe_ids(bpe, token_ids, skip_special_tokens=True):
    ends_word = build_word_end_flags(bpe)
    pieces = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id < 0 or token_id >= len(bpe.itos):
            continue

        if token_id in (0, 1, 2, 3):
            if skip_special_tokens:
                continue
            pieces.append(bpe.itos[token_id])
            continue

        payload = bpe.payload.get(token_id)
        if payload is None:

            if token_id >= 261:
                rank = token_id - 261
                if 0 <= rank < len(bpe.merges):
                    left, right = bpe.merges[rank]
                    payload = bpe.payload.get(left, b"") + bpe.payload.get(right, b"")
            if payload is None:
                payload = b""

        pieces.append(payload.decode("utf-8", errors="replace"))
        if ends_word[token_id]:
            pieces.append(" ")

    return "".join(pieces).strip()

def locate_tokenizer_json():
    candidates = []
    env_path = os.environ.get("BPE_TOKENIZER_PATH")
    if env_path:
        candidates.append(Path(env_path).expanduser())
    candidates.extend([
        TOKENIZER_PATH,
        Path.cwd() / "legal_bpe_tokenizer.json",
        Path.cwd() / "legal_bpe_tokenizer (2).json",
        Path.cwd() / "outputs" / "legal_bpe_tokenizer.json",
        Path("/kaggle/working/legal_bpe_tokenizer.json"),
        Path.home() / "Downloads" / "legal_bpe_tokenizer (2).json",
    ])
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(sorted(kaggle_input.rglob("legal_bpe_tokenizer*.json")))
    seen = set()
    for candidate in candidates:
        try:
            key = str(candidate.resolve())
        except Exception:
            key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find legal_bpe_tokenizer.json. In Kaggle, add the downloaded JSON "
        "as an input dataset (or set BPE_TOKENIZER_PATH to its path). The tokenizer "
        "is intentionally loaded from file and will not be retrained."
    )

TOKENIZER_SOURCE_PATH = locate_tokenizer_json()
with TOKENIZER_SOURCE_PATH.open("r", encoding="utf-8") as f:
    tokenizer_payload = json.load(f)
if int(tokenizer_payload.get("vocab_size", -1)) != TARGET_VOCAB_SIZE:
    raise ValueError(
        f"Tokenizer has vocab_size={tokenizer_payload.get('vocab_size')}; "
        f"expected {TARGET_VOCAB_SIZE}."
    )
tokenizer = ByteBPE.from_json(tokenizer_payload)
if tokenizer.vocab_size != TARGET_VOCAB_SIZE or len(tokenizer.merges) != TARGET_VOCAB_SIZE - 261:
    raise ValueError(
        "The supplied JSON does not contain the complete expected Assignment-1 ByteBPE merge table."
    )
TOKENIZER_PATH.parent.mkdir(parents=True, exist_ok=True)
if TOKENIZER_SOURCE_PATH.resolve() != TOKENIZER_PATH.resolve():
    TOKENIZER_PATH.write_bytes(TOKENIZER_SOURCE_PATH.read_bytes())
TOKENIZER_SHA256 = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]
EOW_ID = 260
SPECIAL_IDS = {PAD_ID, UNK_ID, BOS_ID, EOS_ID}
VOCAB_SIZE = tokenizer.vocab_size
if [tokenizer.itos[i] for i in range(4)] != ["<pad>", "<unk>", "<s>", "</s>"]:
    raise RuntimeError("Assignment-1 reserved-token ordering is invalid.")
if EOW_ID >= VOCAB_SIZE:
    raise RuntimeError("Assignment-1 </w> symbol is missing.")

print("Loaded supplied Assignment-1 ByteBPE:", TOKENIZER_SOURCE_PATH)
print("Tokenizer vocabulary:", VOCAB_SIZE, "| merges:", len(tokenizer.merges))
print("Tokenizer SHA256:", TOKENIZER_SHA256)
sample = "The Supreme Court considered the appeal and dismissed the petition."
sample_ids = tokenizer.encode(sample)
print("Sample token count:", len(sample_ids))
print("Custom round-trip:", decode_bpe_ids(tokenizer, sample_ids))

In [ ]:
TEXT_KEYS = (
    "text", "content", "document", "judgment", "judgement", "body",
    "full_text", "prompt", "prompt_text", "input", "case_text",
)

def extract_text(record):
    if isinstance(record, str):
        return record
    if isinstance(record, dict):
        for key in TEXT_KEYS:
            value = record.get(key)
            if isinstance(value, str) and value.strip():
                return value
        for key, value in record.items():
            if str(key).lower() in {"id", "document_id", "doc_id", "example_id"}:
                continue
            if isinstance(value, str) and len(value.strip()) > 20:
                return value
            if isinstance(value, (dict, list, tuple)):
                found = extract_text(value)
                if found:
                    return found
    if isinstance(record, (list, tuple)):
        for value in record:
            found = extract_text(value)
            if found:
                return found
    return ""

def extract_id(record, fallback):
    if isinstance(record, dict):
        for key in ("id", "ID", "example_id", "document_id", "doc_id"):
            value = record.get(key)
            if value is not None and not isinstance(value, (dict, list)):
                return str(value)

        for key, value in record.items():
            if str(key).strip().isdigit() and isinstance(value, dict) and extract_text(value):
                return str(key)
    return str(fallback)

test_stream = load_dataset(
    DATASET_ID,
    DATASET_CONFIG,
    split="test",
    streaming=True,
    token=HF_TOKEN,
)
test_examples = []
stop_loading = False

for row_index, row in enumerate(
    tqdm(test_stream, desc="Reading official Phase-1 test prompts")
):

    if (
        isinstance(row, dict)
        and row
        and all(str(key).isdigit() for key in row)
    ):
        records = ((str(key), value) for key, value in row.items())
        if row_index == 0:
            print(
                f"Found ID-keyed test object with {len(row)} entries; "
                f"sample IDs: {list(row)[:5]}"
            )
    else:

        records = ((extract_id(row, row_index + 1), row),)
        if row_index == 0:
            fields = list(row.keys())[:10] if isinstance(row, dict) else type(row).__name__
            print("First official test-row fields:", fields)

    for official_id, record in records:
        prompt = extract_text(record).strip()
        if not prompt:
            fields = list(record.keys()) if isinstance(record, dict) else type(record).__name__
            raise ValueError(
                f"Could not extract prompt for official test ID {official_id}; "
                f"available fields={fields}"
            )

        test_examples.append({"id": str(official_id), "prompt": prompt})

        if (
            MAX_TEST_EXAMPLES is not None
            and len(test_examples) >= MAX_TEST_EXAMPLES
        ):
            stop_loading = True
            break

    if stop_loading:
        break

print(f"Loaded official test prompts: {len(test_examples)}")
print("First IDs:", [item["id"] for item in test_examples[:5]])
print("Last IDs:", [item["id"] for item in test_examples[-5:]])

if not test_examples:
    raise RuntimeError("No official test prompts loaded.")

official_ids = [item["id"] for item in test_examples]
if len(set(official_ids)) != len(official_ids):
    raise RuntimeError("Duplicate official test IDs detected.")

prompt_audit = []
for item in test_examples:
    tok_len = len(tokenizer.encode(item["prompt"]))
    prompt_audit.append({
        "id": item["id"],
        "words": len(item["prompt"].split()),
        "tokens": tok_len,
    })

save_json(OUTPUT_DIR / "official_test_prompt_length_audit.json", prompt_audit)

prompt_lengths = [row["tokens"] for row in prompt_audit]
print(f"Official test examples: {len(test_examples)}")
print(f"Prompt words: min={min(row['words'] for row in prompt_audit)}, max={max(row['words'] for row in prompt_audit)}")
print(f"Prompt tokens: min={min(prompt_lengths)}, mean={np.mean(prompt_lengths):.1f}, max={max(prompt_lengths)}")

validation_context_samples = []
MAX_CONTEXT_SAMPLE_DOCS_PER_SOURCE = 64
MAX_CONTEXT_SAMPLE_CHARS = 12_000
for source_name in SOURCE_FILES:
    collected = 0
    for _, _, text, is_valid in iter_bounded_source_documents(source_name):
        if not is_valid:
            continue
        validation_context_samples.append(text[:MAX_CONTEXT_SAMPLE_CHARS])
        collected += 1
        if collected >= MAX_CONTEXT_SAMPLE_DOCS_PER_SOURCE:
            break
heldout_validation_pairs = []
completion_lengths = []
for text in validation_context_samples:
    words = text.split()
    if len(words) >= 512 + OUTPUT_WORDS:
        prompt = " ".join(words[:512])
        reference = " ".join(words[512:512 + OUTPUT_WORDS])
        heldout_validation_pairs.append((prompt, reference))
        completion_lengths.append(len(tokenizer.encode(reference)))
    if len(completion_lengths) >= max(256, LOCAL_EVAL_EXAMPLES):
        break
HELDOUT_VALIDATION_PAIRS = heldout_validation_pairs[:LOCAL_EVAL_EXAMPLES]
del heldout_validation_pairs, validation_context_samples
gc.collect()
if completion_lengths:
    estimated_completion = max(256, int(np.percentile(completion_lengths, 95)))
    print(f"Validation target tokens: median={int(np.median(completion_lengths))}, p95={estimated_completion}, max={max(completion_lengths)}")
else:
    estimated_completion = 256
    print("Could not estimate target token length; using 256-token estimate.")
CONTEXT_ESTIMATED_COMPLETION_TOKENS = estimated_completion
CONTEXT_REQUIRED_TOKENS = max(prompt_lengths) + estimated_completion
print("Longest official prompt tokens:", max(prompt_lengths))
print("Transformer-XL prompt mode: full prompt is processed recurrently without truncation.")
save_json(OUTPUT_DIR / "context_safety_audit.json", {
    "architecture": ARCHITECTURE_NAME,
    "fixed_context_limit": False,
    "prompt_truncation": False,
    "max_official_prompt_tokens": int(max(prompt_lengths)),
    "estimated_completion_tokens": int(CONTEXT_ESTIMATED_COMPLETION_TOKENS),
    "estimated_prompt_plus_completion_tokens": int(CONTEXT_REQUIRED_TOKENS),
    "training_segment_len": int(SEGMENT_LEN),
    "training_memory_len": int(TRAIN_MEM_LEN),
    "generation_memory_len": int(EVAL_MEM_LEN),
})

In [ ]:
def encode_streamed_split(is_validation, token_budget, label):
    per_source_budget = max(1, token_budget // len(SOURCE_FILES))
    source_arrays = []
    split_name = "validation" if is_validation else "train"
    if "stream_split_document_counts" not in globals():
        globals()["stream_split_document_counts"] = {"train": {}, "validation": {}}
    for source_name in SOURCE_FILES:
        ids = np.empty(per_source_budget, dtype=np.int32)
        used = doc_count = 0
        for _, _, text, row_is_validation in tqdm(iter_bounded_source_documents(source_name), desc=f"{label}: {source_name}"):
            if row_is_validation != is_validation:
                continue
            remaining = per_source_budget - used
            if remaining <= 1:
                break
            encoded = tokenizer.encode(text)
            if not encoded:
                continue
            take = min(len(encoded), remaining - 1)
            ids[used:used + take] = np.asarray(encoded[:take], dtype=np.int32)
            used += take
            ids[used] = EOS_ID
            used += 1
            doc_count += 1
            if used >= per_source_budget:
                break
        source_arrays.append(ids[:used].copy())
        stream_split_document_counts[split_name][source_name] = int(doc_count)
        print(f"{label} | {source_name}: docs={doc_count:,}, tokens={used:,}")
    return np.concatenate(source_arrays).astype(np.int32, copy=False)

ENCODED_CACHE_DIR = OUTPUT_DIR / "encoded_training_data"
ENCODED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_IDS_CACHE = ENCODED_CACHE_DIR / "train_ids.npy"
VALID_IDS_CACHE = ENCODED_CACHE_DIR / "valid_ids.npy"
ENCODED_CACHE_METADATA = ENCODED_CACHE_DIR / "metadata.json"
ENCODED_DATA_SPEC = {
    "dataset_id": DATASET_ID, "dataset_config": DATASET_CONFIG,
    "source_files": SOURCE_FILES, "seed": SEED,
    "validation_fraction": VALID_FRACTION, "max_train_docs": MAX_TRAIN_DOCS,
    "max_corpus_chars": MAX_CORPUS_CHARS, "max_chars_per_doc": MAX_CHARS_PER_DOC,
    "train_token_budget": TRAIN_TOKEN_BUDGET, "valid_token_budget": VALID_TOKEN_BUDGET,
    "tokenizer_sha256": TOKENIZER_SHA256,
}
ENCODED_DATA_SIGNATURE = stable_json_hash(ENCODED_DATA_SPEC)
cache_loaded = False
if TRAIN_IDS_CACHE.exists() and VALID_IDS_CACHE.exists() and ENCODED_CACHE_METADATA.exists():
    try:
        cache_metadata = json.loads(ENCODED_CACHE_METADATA.read_text(encoding="utf-8"))
        if cache_metadata.get("signature") == ENCODED_DATA_SIGNATURE:
            train_ids = np.load(TRAIN_IDS_CACHE, mmap_mode="r")
            valid_ids = np.load(VALID_IDS_CACHE, mmap_mode="r")
            stream_split_document_counts = cache_metadata["document_counts"]
            cache_loaded = True
            print("Reusing compatible memory-mapped token arrays.")
    except Exception as exc:
        print("Encoded cache could not be reused; rebuilding:", exc)
if not cache_loaded:
    stream_split_document_counts = {"train": {}, "validation": {}}
    train_ids = encode_streamed_split(False, TRAIN_TOKEN_BUDGET, "Encoding train")
    valid_ids = encode_streamed_split(True, VALID_TOKEN_BUDGET, "Encoding validation")
    ENCODED_CACHE_METADATA.unlink(missing_ok=True)
    train_tmp = ENCODED_CACHE_DIR / "train_ids.pending.npy"
    valid_tmp = ENCODED_CACHE_DIR / "valid_ids.pending.npy"
    np.save(train_tmp, train_ids)
    np.save(valid_tmp, valid_ids)
    train_tmp.replace(TRAIN_IDS_CACHE)
    valid_tmp.replace(VALID_IDS_CACHE)
    save_json(ENCODED_CACHE_METADATA, {
        "signature": ENCODED_DATA_SIGNATURE,
        "document_counts": stream_split_document_counts,
        "train_tokens": int(len(train_ids)), "valid_tokens": int(len(valid_ids)),
    })
if len(train_ids) < SEGMENT_LEN + 1 or len(valid_ids) < SEGMENT_LEN + 1:
    raise RuntimeError("Training or validation split is too short for one segment.")
TRAIN_DOCUMENT_COUNT = int(sum(stream_split_document_counts["train"].values()))
VALID_DOCUMENT_COUNT = int(sum(stream_split_document_counts["validation"].values()))
print("Train tokens:", len(train_ids), "| validation tokens:", len(valid_ids))
print("Train docs:", TRAIN_DOCUMENT_COUNT, "| validation docs:", VALID_DOCUMENT_COUNT)

class SequentialSegmentStream:
    def __init__(self, token_ids, segment_len, batch_size=1):
        ids = np.asarray(token_ids, dtype=np.int32)
        self.segment_len = int(segment_len)
        self.batch_size = int(batch_size)
        self.lane_length = (len(ids) - 1) // self.batch_size
        if self.lane_length < 1:
            raise ValueError("Token stream is too short for the batch size.")
        lanes = np.empty((self.batch_size, self.lane_length + 1), dtype=np.int64)
        for lane in range(self.batch_size):
            start = lane * self.lane_length
            lanes[lane] = ids[start:start + self.lane_length + 1]
        self.data = torch.from_numpy(lanes)
    def __len__(self):
        return math.ceil(self.lane_length / self.segment_len)
    def __iter__(self):
        for start in range(0, self.lane_length, self.segment_len):
            end = min(start + self.segment_len, self.lane_length)
            yield self.data[:, start:end].contiguous(), self.data[:, start + 1:end + 1].contiguous()

train_stream = SequentialSegmentStream(train_ids, SEGMENT_LEN, MICRO_BATCH_SIZE)
valid_stream = SequentialSegmentStream(valid_ids, SEGMENT_LEN, MICRO_BATCH_SIZE)
print("Sequential train/validation segments:", len(train_stream), len(valid_stream))

In [ ]:
def sinusoidal_relative_positions(max_distance, d_model, device):
    positions = torch.arange(max_distance + 1, device=device, dtype=torch.float32)
    inv_freq = torch.exp(-math.log(RELATIVE_POSITION_BASE) * torch.arange(0, d_model, 2, device=device, dtype=torch.float32) / d_model)
    angles = positions[:, None] * inv_freq[None, :]
    return torch.cat((angles.sin(), angles.cos()), dim=-1)

class RelativeMultiheadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.d_model, self.n_heads = d_model, n_heads
        self.head_dim = d_model // n_heads
        self.attn_dropout = nn.Dropout(dropout)
        self.q_net = nn.Linear(d_model, d_model, bias=False)
        self.k_net = nn.Linear(d_model, d_model, bias=False)
        self.v_net = nn.Linear(d_model, d_model, bias=False)
        self.r_net = nn.Linear(d_model, d_model, bias=False)
        self.o_net = nn.Linear(d_model, d_model, bias=False)
        self.r_w_bias = nn.Parameter(torch.zeros(n_heads, self.head_dim))
        self.r_r_bias = nn.Parameter(torch.zeros(n_heads, self.head_dim))
    def _split_heads(self, x):
        b, t, _ = x.shape
        return x.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
    def forward(self, x, memory=None):
        b, qlen, _ = x.shape
        mlen = 0 if memory is None else memory.size(1)
        kv_input = x if memory is None else torch.cat((memory, x), dim=1)
        klen = kv_input.size(1)
        q = self._split_heads(self.q_net(x))
        k = self._split_heads(self.k_net(kv_input))
        value = self._split_heads(self.v_net(kv_input))
        ac = torch.matmul((q + self.r_w_bias[None, :, None, :]).float(), k.float().transpose(-1, -2))

        max_distance = mlen + qlen - 1
        rel_pos = sinusoidal_relative_positions(max_distance, self.d_model, x.device)
        rel_keys = self._split_heads(self.r_net(rel_pos[None, :, :]))[0].transpose(0, 1)
        bd_by_distance = torch.einsum(
            "bhqd,rhd->bhqr",
            (q + self.r_r_bias[None, :, None, :]).float(),
            rel_keys.float(),
        )
        key_pos = torch.arange(-mlen, qlen, device=x.device)
        query_pos = torch.arange(qlen, device=x.device)
        rel_dist = (query_pos[:, None] - key_pos[None, :]).clamp_min(0)
        gather_index = rel_dist[None, None].expand(b, self.n_heads, qlen, klen)
        bd = torch.gather(bd_by_distance, dim=-1, index=gather_index)
        scores = (ac + bd) * (self.head_dim ** -0.5)
        causal = key_pos[None, :] <= query_pos[:, None]
        scores = scores.masked_fill(~causal[None, None], torch.finfo(scores.dtype).min)
        weights = torch.softmax(scores, dim=-1).to(value.dtype)
        weights = self.attn_dropout(weights)
        output = torch.matmul(weights, value).transpose(1, 2).contiguous().view(b, qlen, self.d_model)
        return self.o_net(output)

class FeedForward(nn.Module):
    def __init__(self, d_model, hidden, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, hidden), nn.GELU(), nn.Linear(hidden, d_model), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class TransformerXLBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_hidden, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn = RelativeMultiheadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, ff_hidden, dropout)
        self.resid_dropout = nn.Dropout(dropout)
    def forward(self, x, memory=None):
        x = x + self.resid_dropout(self.attn(self.norm1(x), memory))
        return x + self.resid_dropout(self.ff(self.norm2(x)))

class TransformerXLLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, ff_hidden, mem_len, dropout):
        super().__init__()
        self.vocab_size, self.d_model = int(vocab_size), int(d_model)
        self.n_layers, self.mem_len = int(n_layers), int(mem_len)
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.embedding_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([TransformerXLBlock(d_model, n_heads, ff_hidden, dropout) for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self.apply(self._init_weights)
        nn.init.normal_(self.token_embedding.weight, mean=0.0, std=0.02)
        for layer in self.layers:
            nn.init.normal_(layer.attn.r_w_bias, mean=0.0, std=0.02)
            nn.init.normal_(layer.attn.r_r_bias, mean=0.0, std=0.02)

        residual_std = 0.02 / math.sqrt(2 * n_layers)
        for layer in self.layers:
            nn.init.normal_(layer.attn.o_net.weight, mean=0.0, std=residual_std)
            nn.init.normal_(layer.ff.net[2].weight, mean=0.0, std=residual_std)
    @staticmethod
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding): nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)
    def _update_mems(self, hidden_states, mems, mem_len):
        result = []
        for level, current in enumerate(hidden_states):
            old = None if mems is None else mems[level]
            joined = current.detach() if old is None else torch.cat((old.detach(), current.detach()), dim=1)
            result.append(joined[:, -mem_len:, :].detach() if mem_len > 0 else None)
        return tuple(result)
    def forward(self, input_ids, mems=None, mem_len=None):
        if input_ids.ndim != 2: raise ValueError("input_ids must have shape [batch, segment_length]")
        if mems is not None and len(mems) != self.n_layers + 1:
            raise ValueError(f"Expected {self.n_layers + 1} memory levels, got {len(mems)}")
        keep = self.mem_len if mem_len is None else int(mem_len)
        hidden = self.embedding_dropout(self.token_embedding(input_ids))
        hidden_states = [hidden]
        for i, layer in enumerate(self.layers):
            layer_memory = None if mems is None else mems[i]
            if self.training and USE_GRAD_CHECKPOINTING:
                hidden = checkpoint(layer, hidden, layer_memory, use_reentrant=False)
            else:
                hidden = layer(hidden, layer_memory)
            hidden_states.append(hidden)
        logits = self.lm_head(self.final_norm(hidden))
        return logits, self._update_mems(hidden_states, mems, keep)

model_config = {
    "architecture": ARCHITECTURE_NAME,
    "positional_encoding": "Transformer-XL relative sinusoidal",
    "segment_recurrence": True,
    "memories_detached_between_segments": True,
    "content_position_biases": "learned u and v",
    "separate_content_and_relative_key_projections": True,
    "vocab_size": VOCAB_SIZE, "d_model": MODEL_D, "heads": MODEL_HEADS,
    "head_dim": MODEL_D // MODEL_HEADS, "layers": MODEL_LAYERS,
    "ff_hidden": MODEL_FF, "training_segment_len": SEGMENT_LEN,
    "training_mem_len": TRAIN_MEM_LEN, "generation_mem_len": EVAL_MEM_LEN,
    "dropout": MODEL_DROPOUT, "weight_tied_input_output": True,
}
def build_model():
    return TransformerXLLanguageModel(VOCAB_SIZE, MODEL_D, MODEL_HEADS, MODEL_LAYERS, MODEL_FF, TRAIN_MEM_LEN, MODEL_DROPOUT)
def count_parameters(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)
def architecture_audit(model):
    total = count_parameters(model)
    return {
        "total_parameters": int(total), "trainable_parameters": int(total),
        "parameter_millions": total / 1e6,
        "weight_tied_input_output": model.lm_head.weight.data_ptr() == model.token_embedding.weight.data_ptr(),
        "learned_absolute_position_parameters": 0,
        "positional_encoding": "Transformer-XL relative sinusoidal",
        "segment_recurrence": True, "memories_detached_between_segments": True,
        "config": model_config,
    }
model_for_audit = build_model()
audit = architecture_audit(model_for_audit)
print(json.dumps(audit, indent=2))
if not (70_000_000 <= audit["total_parameters"] <= 250_000_000):
    raise RuntimeError(f"Model has {audit['total_parameters']:,} parameters; required range is 70M-250M.")
if not audit["weight_tied_input_output"]: raise RuntimeError("Token embedding and LM head must be tied.")
print("Transformer-XL uses content + relative score terms with learned u/v biases.")
del model_for_audit
gc.collect()

In [ ]:
if RUN_SMOKE_TEST:
    smoke_model = build_model().to(DEVICE)
    smoke_model.train()
    smoke_optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=LEARNING_RATE)
    smoke_optimizer.zero_grad(set_to_none=True)
    n = min(32, SEGMENT_LEN)
    sx1 = torch.randint(0, VOCAB_SIZE, (MICRO_BATCH_SIZE, n), device=DEVICE)
    sy1 = torch.randint(0, VOCAB_SIZE, (MICRO_BATCH_SIZE, n), device=DEVICE)
    sx2 = torch.randint(0, VOCAB_SIZE, (MICRO_BATCH_SIZE, n), device=DEVICE)
    sy2 = torch.randint(0, VOCAB_SIZE, (MICRO_BATCH_SIZE, n), device=DEVICE)
    enabled = DEVICE.type == "cuda" and USE_AMP
    dtype = torch.bfloat16 if enabled and torch.cuda.is_bf16_supported() else torch.float16
    scaler = make_grad_scaler(enabled and dtype == torch.float16)
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.autocast(device_type="cuda", dtype=dtype, enabled=enabled):
        l1, m1 = smoke_model(sx1, mem_len=TRAIN_MEM_LEN)
        l2, m2 = smoke_model(sx2, mems=m1, mem_len=TRAIN_MEM_LEN)
        loss = (F.cross_entropy(l1.reshape(-1, VOCAB_SIZE), sy1.reshape(-1)) + F.cross_entropy(l2.reshape(-1, VOCAB_SIZE), sy2.reshape(-1))) / 2
    if scaler is not None and scaler.is_enabled():
        scaler.scale(loss).backward()
        scaler.unscale_(smoke_optimizer)
        torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), GRAD_CLIP)
        scaler.step(smoke_optimizer)
        scaler.update()
    else:
        loss.backward()
        torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), GRAD_CLIP)
        smoke_optimizer.step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        print("Peak VRAM (GB):", round(torch.cuda.max_memory_allocated() / (1024**3), 2))
    print("Two-segment loss:", round(float(loss.item()), 4))
    print("Memory levels:", len(m2), "| top-layer memory tokens:", m2[-1].size(1))
    print("Smoke update seconds:", round(time.time() - t0, 2))
    del smoke_model, smoke_optimizer, sx1, sy1, sx2, sy2, l1, l2, m1, m2, loss, scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
USE_AMP = DEVICE.type == "cuda" and USE_AMP
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16

def amp_context():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)

def make_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(step):
        if step < warmup_steps: return max(1e-8, (step + 1) / max(1, warmup_steps))
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * min(1.0, max(0.0, progress)))))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

@torch.no_grad()
def validation_loss(model, stream, max_batches=MAX_EVAL_BATCHES):
    model.eval()
    mems = None
    total_loss = 0.0
    total_tokens = 0
    for i, (x, y) in enumerate(stream):
        if i >= max_batches: break
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with amp_context():
            logits, mems = model(x, mems=mems, mem_len=TRAIN_MEM_LEN)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))
        mems = tuple(m.detach() if m is not None else None for m in mems)
        n = int(y.numel())
        total_loss += float(loss.item()) * n
        total_tokens += n
    if not total_tokens: raise RuntimeError("Validation stream produced no tokens.")
    return total_loss / total_tokens

def save_checkpoint(path, model, optimizer, scheduler, scaler, epoch, step, best_valid_loss, history):
    torch.save({
        "architecture": ARCHITECTURE_NAME, "model_config": model_config,
        "model_state_dict": model.state_dict(),
        "epoch": int(epoch), "step": int(step),
        "best_valid_loss": float(best_valid_loss), "history": history,
        "seed": SEED, "vocab_size": VOCAB_SIZE,
        "tokenizer_sha256": TOKENIZER_SHA256,
    }, path)

In [ ]:
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
SEGMENTS_PER_EPOCH = len(train_stream)
STEPS_PER_EPOCH = max(1, math.ceil(SEGMENTS_PER_EPOCH / GRAD_ACCUM))
MAX_STEPS = EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = max(1, int(round(MAX_STEPS * WARMUP_FRACTION)))
TOKENS_PER_OPTIMIZER_STEP = MICRO_BATCH_SIZE * GRAD_ACCUM * SEGMENT_LEN
print("GPU:", GPU_NAME)
print("Train tokens:", f"{len(train_ids):,}", "| sequential segments:", SEGMENTS_PER_EPOCH)
print("Segment length / train memory:", SEGMENT_LEN, "/", TRAIN_MEM_LEN)
print("Tokens per full optimizer update:", TOKENS_PER_OPTIMIZER_STEP)
print("Epochs:", EPOCHS, "| updates/epoch:", STEPS_PER_EPOCH, "| total updates:", MAX_STEPS)
print("Approx token presentations:", f"{TRAIN_TOKEN_BUDGET * EPOCHS:,}")

In [ ]:
def try_load_warm_start(model):
    if not CONTINUE_FROM_EXISTING_CHECKPOINT or not BEST_CHECKPOINT_PATH.exists(): return False
    try:
        payload = torch.load(BEST_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
        compatible = (
            payload.get("architecture") == ARCHITECTURE_NAME
            and payload.get("vocab_size") == VOCAB_SIZE
            and payload.get("tokenizer_sha256") == TOKENIZER_SHA256
            and payload.get("model_config") == model_config
        )
        if not compatible:
            print("Checkpoint is incompatible; starting fresh.")
            return False
        state = payload["model_state_dict"]
        finite = all(
            torch.isfinite(tensor).all().item()
            for tensor in state.values()
            if torch.is_tensor(tensor) and tensor.is_floating_point()
        )
        if not finite:
            print("Checkpoint contains non-finite weights; ignoring it and starting fresh.")
            return False
        model.load_state_dict(state)
        print("Warm-started from Transformer-XL checkpoint:", BEST_CHECKPOINT_PATH)
        return True
    except Exception as exc:
        print("Could not warm-start; starting fresh:", exc)
        return False

def train_model():
    if DEVICE.type != "cuda": raise RuntimeError("Select the Kaggle GPU accelerator before training.")
    model = build_model().to(DEVICE)
    warm_started = try_load_warm_start(model)
    run_lr = WARM_START_LEARNING_RATE if warm_started else LEARNING_RATE
    decay_params, no_decay_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if param.ndim < 2 or "embedding" in name or name.endswith(("r_w_bias", "r_r_bias")):
            no_decay_params.append(param)
        else:
            decay_params.append(param)
    optimizer = torch.optim.AdamW(
        [{"params": decay_params, "weight_decay": WEIGHT_DECAY},
         {"params": no_decay_params, "weight_decay": 0.0}],
        lr=run_lr, betas=(0.9, 0.95), eps=1e-8,
    )
    scheduler = make_scheduler(optimizer, MAX_STEPS, WARMUP_STEPS)
    scaler = make_grad_scaler(USE_SCALER) if USE_SCALER else None
    criterion = nn.CrossEntropyLoss()
    history = []
    best_loss = validation_loss(model, valid_stream) if warm_started else float("inf")
    if warm_started: print(f"Warm-start validation loss={best_loss:.4f}")
    step = tokens_seen = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        iterator = iter(train_stream)
        batch_count = len(train_stream)
        mems = None
        epoch_loss_sum = epoch_token_count = 0
        for group_start in range(0, batch_count, GRAD_ACCUM):
            group_size = min(GRAD_ACCUM, batch_count - group_start)
            optimizer.zero_grad(set_to_none=True)
            group_loss = 0.0
            for _ in range(group_size):
                x, y = next(iterator)
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                with amp_context():
                    logits, next_mems = model(x, mems=mems, mem_len=TRAIN_MEM_LEN)
                    loss = criterion(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))
                    if not torch.isfinite(loss).item():
                        optimizer.zero_grad(set_to_none=True)
                        raise FloatingPointError(
                            f"Non-finite training loss at epoch {epoch}, step {step}. "
                            "The update was stopped; use the reduced learning rate and restart from a finite checkpoint."
                        )
                    scaled_loss = loss / group_size
                if scaler is not None and scaler.is_enabled(): scaler.scale(scaled_loss).backward()
                else: scaled_loss.backward()
                mems = tuple(m.detach() if m is not None else None for m in next_mems)
                group_loss += float(loss.item())
                count = int(x.numel())
                epoch_loss_sum += float(loss.item()) * count
                epoch_token_count += count
                tokens_seen += count
            if scaler is not None and scaler.is_enabled(): scaler.unscale_(optimizer)
            grad_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP).item())
            if not math.isfinite(grad_norm):
                print(f"Skipping update {step + 1}: non-finite gradient norm; resetting recurrent memory.")
                if scaler is not None and scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    for group in optimizer.param_groups:
                        group["lr"] *= 0.5
                optimizer.zero_grad(set_to_none=True)
                mems = None
                step += 1
                continue
            if scaler is not None and scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else: optimizer.step()
            scheduler.step()
            step += 1
            mean_loss = group_loss / group_size
            if step == 1 or step % 25 == 0:
                print(f"epoch={epoch}/{EPOCHS} step={step}/{MAX_STEPS} loss={mean_loss:.4f} lr={optimizer.param_groups[0]['lr']:.3e} grad={grad_norm:.3f} tokens={tokens_seen:,}")
            if step % EVAL_EVERY_STEPS == 0 or step == MAX_STEPS:
                val_loss = validation_loss(model, valid_stream)
                if not math.isfinite(val_loss):
                    raise FloatingPointError(
                        f"Non-finite validation loss at epoch {epoch}, step {step}; "
                        "the checkpoint and JSON history were not updated."
                    )
                val_ppl = math.exp(min(val_loss, 20.0))
                if val_loss < best_loss:
                    best_loss = val_loss
                    save_checkpoint(BEST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler, epoch, step, best_loss, history)
                history.append({"epoch": epoch, "step": step, "train_loss": mean_loss,
                    "valid_loss": val_loss, "valid_perplexity": val_ppl,
                    "learning_rate": float(optimizer.param_groups[0]["lr"]),
                    "grad_norm": grad_norm, "tokens_processed": tokens_seen})
                save_checkpoint(LATEST_CHECKPOINT_PATH, model, optimizer, scheduler, scaler, epoch, step, best_loss, history)
                save_json(OUTPUT_DIR / "training_history.json", history)
                print(f"  validation loss={val_loss:.4f}, perplexity={val_ppl:.2f}, best={best_loss:.4f}")
        print(f"Epoch {epoch}/{EPOCHS} | token-weighted loss={epoch_loss_sum / max(1, epoch_token_count):.4f}")

    if not BEST_CHECKPOINT_PATH.exists(): raise RuntimeError("No best checkpoint was saved.")
    summary = {
        "architecture": ARCHITECTURE_NAME, "model_config": model_config,
        "parameters": int(count_parameters(model)), "epochs_completed": int(EPOCHS),
        "warm_started_from_existing_checkpoint": bool(warm_started),
        "learning_rate": float(run_lr), "steps_completed": int(step),
        "tokens_processed_this_run": int(tokens_seen), "best_valid_loss": float(best_loss),
        "best_validation_perplexity": float(math.exp(min(best_loss, 20.0))),
        "best_checkpoint": str(BEST_CHECKPOINT_PATH),
        "latest_checkpoint": str(LATEST_CHECKPOINT_PATH), "training_complete": True,
    }
    save_json(OUTPUT_DIR / "training_summary.json", summary)
    del model, optimizer, scheduler, scaler, iterator, mems
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

if RUN_TRAINING:
    training_summary = train_model()
    print(json.dumps(training_summary, indent=2))
else:
    summary_path = OUTPUT_DIR / "training_summary.json"
    training_summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else None
    if training_summary is None: print("No completed run found; set RUN_TRAINING=True.")

In [ ]:
def load_selected_model():
    path = BEST_CHECKPOINT_PATH if BEST_CHECKPOINT_PATH.exists() else LATEST_CHECKPOINT_PATH
    if not path.exists(): raise FileNotFoundError("Train Transformer-XL first; no checkpoint found.")
    payload = torch.load(path, map_location=DEVICE, weights_only=False)
    if payload.get("architecture") != ARCHITECTURE_NAME: raise ValueError("Checkpoint architecture mismatch.")
    if payload.get("vocab_size") != VOCAB_SIZE: raise ValueError("Checkpoint vocabulary mismatch.")
    if payload.get("tokenizer_sha256") != TOKENIZER_SHA256: raise ValueError("Checkpoint tokenizer mismatch.")
    model = build_model().to(DEVICE)
    model.load_state_dict(payload["model_state_dict"])
    model.eval()
    return model, path

def sample_token(logits, temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P):
    logits = logits.float().clone()
    for token_id in SPECIAL_IDS:
        if 0 <= int(token_id) < logits.numel(): logits[int(token_id)] = -float("inf")
    if temperature <= 0: return int(torch.argmax(logits))
    logits = logits / temperature
    if top_k and 0 < top_k < logits.numel():
        cutoff = torch.topk(logits, top_k).values[-1]
        logits[logits < cutoff] = -float("inf")
    if top_p and 0 < top_p < 1.0:
        vals, inds = torch.sort(logits, descending=True)
        probs = torch.softmax(vals, dim=-1)
        remove = probs.cumsum(-1) > top_p
        remove[1:] = remove[:-1].clone()
        remove[0] = False
        logits[inds[remove]] = -float("inf")
    probs = torch.softmax(logits, dim=-1)
    if not torch.isfinite(probs).all() or float(probs.sum()) <= 0: return int(torch.argmax(logits))
    return int(torch.multinomial(probs, 1))

@torch.no_grad()
def generate_continuation(model, prompt, target_words=OUTPUT_WORDS, max_tokens=None, temperature=None):
    model.eval()
    temperature = TEMPERATURE if temperature is None else temperature
    max_tokens = MAX_GENERATION_TOKENS if max_tokens is None else int(max_tokens)
    prompt_ids = tokenizer.encode(prompt) or [BOS_ID]
    mems, logits = None, None
    for start in range(0, len(prompt_ids), SEGMENT_LEN):
        chunk = torch.tensor([prompt_ids[start:start + SEGMENT_LEN]], dtype=torch.long, device=DEVICE)
        with amp_context(): logits, mems = model(chunk, mems=mems, mem_len=EVAL_MEM_LEN)
    next_logits = logits[0, -1]
    end_flags = build_word_end_flags(tokenizer)
    generated, completed = [], 0
    current_has_bytes, pieces = False, 0
    for _ in range(max_tokens):
        candidate = next_logits
        if not current_has_bytes:
            candidate = candidate.clone()
            candidate[EOW_ID] = -float("inf")
        token_id = sample_token(candidate, temperature=temperature)
        generated.append(token_id)
        pending_ids = [token_id]
        payload = tokenizer.payload.get(token_id, b"")
        if end_flags[token_id]:
            if current_has_bytes or payload: completed += 1
            current_has_bytes, pieces = False, 0
        else:
            if payload: current_has_bytes = True
            pieces += 1
            if current_has_bytes and pieces >= MAX_BPE_TOKENS_PER_WORD:
                generated.append(EOW_ID)
                pending_ids.append(EOW_ID)
                completed += 1
                current_has_bytes, pieces = False, 0
        if completed >= target_words:
            words = decode_bpe_ids(tokenizer, generated, skip_special_tokens=True).strip().split()
            if len(words) < target_words:
                raise RuntimeError(f"Boundary accounting yielded {len(words)} decoded words; expected {target_words}.")
            return " ".join(words[:target_words])
        pending = torch.tensor([pending_ids], dtype=torch.long, device=DEVICE)
        with amp_context(): logits, mems = model(pending, mems=mems, mem_len=EVAL_MEM_LEN)
        next_logits = logits[0, -1]
    count = len(decode_bpe_ids(tokenizer, generated, skip_special_tokens=True).split())
    raise RuntimeError(f"Generated {count} words (< {target_words}) after {max_tokens} sampled tokens.")

def generate_128_word_continuation(model, prompt, temperature=None):
    last_error = None
    for budget in GENERATION_TOKEN_BUDGETS:
        try:
            result = generate_continuation(model, prompt, target_words=OUTPUT_WORDS, max_tokens=budget, temperature=temperature)
            if len(result.split()) != OUTPUT_WORDS: raise RuntimeError("Unexpected generated word count.")
            return result
        except RuntimeError as exc:
            last_error = exc
            if "after " not in str(exc).lower() and "expected" not in str(exc).lower(): raise
    raise RuntimeError(f"Could not generate {OUTPUT_WORDS} words with budgets {GENERATION_TOKEN_BUDGETS}. Last error: {last_error}") from last_error

def validation_pairs(limit=LOCAL_EVAL_EXAMPLES):
    pairs = HELDOUT_VALIDATION_PAIRS[:limit]
    if not pairs: raise RuntimeError("No held-out validation prompt/reference pairs are available.")
    return pairs

def lcs_length(a, b):
    if not a or not b:
        return 0
    if len(a) < len(b):
        a, b = b, a
    masks = {}
    for j, token in enumerate(a):
        masks[token] = masks.get(token, 0) | (1 << j)
    state = 0
    for token in b:
        m = masks.get(token, 0)
        x = m | state
        y = (state << 1) | 1
        state = x & ~(x - y)
    return state.bit_count() if hasattr(int, "bit_count") else bin(state).count("1")

def rouge_l_f1(pred_text, ref_text):
    pred = pred_text.split()
    ref = ref_text.split()
    if not pred or not ref:
        return 0.0
    l = lcs_length(pred, ref)
    if not l:
        return 0.0
    precision = l / len(pred)
    recall = l / len(ref)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

def ngrams(tokens, n):
    if len(tokens) < n:
        return Counter()
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def bleu4(preds, refs):
    order = 4
    clipped = [0] * order
    totals = [0] * order
    pred_len = 0
    ref_len = 0

    for pred, ref in zip(preds, refs):
        pred_len += len(pred)
        ref_len += len(ref)
        for n in range(1, order + 1):
            pn = ngrams(pred, n)
            rn = ngrams(ref, n)
            totals[n - 1] += sum(pn.values())
            for gram, count in pn.items():
                clipped[n - 1] += min(count, rn.get(gram, 0))

    precisions = [
        clipped[i] / totals[i] if totals[i] else 0.0
        for i in range(order)
    ]
    if any(x <= 0 for x in precisions) or pred_len == 0:
        return 0.0

    bp = 1.0 if pred_len > ref_len else math.exp(1.0 - ref_len / pred_len)
    return bp * math.exp(sum(math.log(x) for x in precisions) / order)

def local_bertscore(pred_texts, ref_texts):
    import bert_score
    device_name = "cuda" if torch.cuda.is_available() else "cpu"
    _, _, f1 = bert_score.score(
        pred_texts,
        ref_texts,
        lang="en",
        batch_size=8,
        verbose=False,
        device=device_name,
    )
    return float(f1.mean().item())

In [ ]:
print("Training segment length:", SEGMENT_LEN)
print("Training memory length:", TRAIN_MEM_LEN)
print("Generation memory length:", EVAL_MEM_LEN)
print("Generation token retry budgets:", GENERATION_TOKEN_BUDGETS)
print("Prompt truncation:", False)

In [ ]:
TRAINING_COMPLETE_FOR_OUTPUTS = bool(
    training_summary is not None and training_summary.get("training_complete", False)
)

if RUN_LOCAL_GENERATION_EVAL and TRAINING_COMPLETE_FOR_OUTPUTS:
    model, checkpoint_path = load_selected_model()
    pairs = validation_pairs()

    local_eval_dir = OUTPUT_DIR / "local_eval"
    local_eval_dir.mkdir(parents=True, exist_ok=True)
    truth = {f"val_{i+1:04d}": {"text": ref} for i, (_, ref) in enumerate(pairs)}
    predictions = {}
    examples = {}

    for i, (prompt, reference) in enumerate(tqdm(pairs, desc="Local held-out generation")):
        prediction = generate_128_word_continuation(model, prompt)
        key = f"val_{i+1:04d}"
        predictions[key] = {"text": prediction}
        examples[key] = {"prompt": prompt, "reference": reference, "prediction": prediction}

    save_json(local_eval_dir / "truth.json", truth)
    save_json(local_eval_dir / "prediction.json", predictions)
    save_json(local_eval_dir / "examples.json", examples)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    pred_texts = [predictions[key]["text"].strip() for key in truth]
    ref_texts = [truth[key]["text"].strip() for key in truth]
    local_metrics = {
        "architecture": ARCHITECTURE_NAME,
        "checkpoint": str(checkpoint_path),
        "examples": len(truth),
        "BERTScore_F1": local_bertscore(pred_texts, ref_texts),
        "ROUGE_L_F1": float(np.mean([rouge_l_f1(p, r) for p, r in zip(pred_texts, ref_texts)])),
        "BLEU_4": bleu4([p.split() for p in pred_texts], [r.split() for r in ref_texts]),
    }
    save_json(local_eval_dir / "metrics.json", local_metrics)
    print(json.dumps(local_metrics, indent=2))
elif RUN_LOCAL_GENERATION_EVAL:
    print("Local generation evaluation skipped: train the model first.")

In [ ]:
if RUN_DECODING_SWEEP and TRAINING_COMPLETE_FOR_OUTPUTS:
    sweep_pairs = validation_pairs(limit=LOCAL_EVAL_EXAMPLES)
    sweep_model, sweep_checkpoint = load_selected_model()
    sweep_predictions = {}

    sweep_configs = [(str(t), float(t), int(EVAL_MEM_LEN)) for t in DECODING_TEMPERATURES]
    for m in EVAL_MEM_LEN_OPTIONS:
        if int(m) != int(EVAL_MEM_LEN):
            sweep_configs.append((f"mem{int(m)}_t0.8", 0.8, int(m)))
    default_mem_len = int(EVAL_MEM_LEN)

    for key, temperature, mem_len in sweep_configs:
        EVAL_MEM_LEN = mem_len
        outputs = []
        for prompt, _ in tqdm(
            sweep_pairs,
            desc=f"Validation decoding sweep ({key}, memory={mem_len})",
        ):
            outputs.append(
                generate_128_word_continuation(
                    sweep_model,
                    prompt,
                    temperature=temperature,
                )
            )
        sweep_predictions[key] = outputs
    EVAL_MEM_LEN = default_mem_len

    del sweep_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    sweep_results = {}
    sweep_references = [reference for _, reference in sweep_pairs]
    for key, outputs in sweep_predictions.items():
        sweep_results[key] = {
            "BERTScore_F1": local_bertscore(outputs, sweep_references),
            "ROUGE_L_F1": float(np.mean([
                rouge_l_f1(pred, ref)
                for pred, ref in zip(outputs, sweep_references)
            ])),
            "BLEU_4": bleu4(
                [pred.split() for pred in outputs],
                [ref.split() for ref in sweep_references],
            ),
        }

    best_key = max(
        sweep_results,
        key=lambda key: sweep_results[key]["BERTScore_F1"],
    )
    _cfg = {k: (t, m) for k, t, m in sweep_configs}
    TEMPERATURE, EVAL_MEM_LEN = _cfg[best_key][0], _cfg[best_key][1]
    best_temperature_key = str(TEMPERATURE)
    sweep_payload = {
        "checkpoint": str(sweep_checkpoint),
        "examples": len(sweep_pairs),
        "selected_config": best_key,
        "selected_temperature_by_BERTScore_F1": TEMPERATURE,
        "selected_generation_memory_len": int(EVAL_MEM_LEN),
        "metrics_by_temperature": sweep_results,
        "predictions_by_temperature": sweep_predictions,
    }
    save_json(OUTPUT_DIR / "local_eval" / "decoding_temperature_sweep.json", sweep_payload)
    print(json.dumps({
        "selected_config": best_key,
        "selected_temperature_by_BERTScore_F1": TEMPERATURE,
        "selected_generation_memory_len": int(EVAL_MEM_LEN),
        "metrics_by_temperature": sweep_results,
    }, indent=2))
elif RUN_DECODING_SWEEP:
    print("Decoding sweep skipped: train the model first.")

In [ ]:
def validate_prediction_file(path, expected_ids):
    path = Path(path)

    def pairs_no_duplicates(pairs):
        result = {}
        duplicates = []
        for key, value in pairs:
            if key in result:
                duplicates.append(key)
            result[key] = value
        if duplicates:
            raise ValueError(f"Duplicate JSON keys: {duplicates[:10]}")
        return result

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(
            f,
            object_pairs_hook=pairs_no_duplicates,
            parse_constant=lambda value: (_ for _ in ()).throw(ValueError(f"Non-standard JSON value: {value}")),
        )

    if not isinstance(data, dict):
        raise ValueError("prediction.json must be a JSON object.")
    if set(data) != set(expected_ids):
        raise ValueError(
            f"ID mismatch: missing={len(set(expected_ids)-set(data))}, extra={len(set(data)-set(expected_ids))}"
        )

    for key in expected_ids:
        rec = data[key]
        if not isinstance(rec, dict) or "text" not in rec:
            raise ValueError(f"Invalid record for ID {key!r}")
        text = rec["text"]
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f"Empty/non-string text for ID {key!r}")
        n_words = len(text.split())
        if n_words != OUTPUT_WORDS:
            raise ValueError(f"ID {key} contains {n_words} words; expected {OUTPUT_WORDS}")

    return data

def write_submission(predictions):
    sub_dir = OUTPUT_DIR / "submission_transformer_xl"
    sub_dir.mkdir(parents=True, exist_ok=True)
    json_path = sub_dir / "prediction.json"
    payload = {key: {"text": predictions[key]} for key in official_ids}
    save_json(json_path, payload)
    validate_prediction_file(json_path, official_ids)

    zip_path = OUTPUT_DIR / "submission_phase1_transformer_xl.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(json_path, arcname="prediction.json")

    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        if names != ["prediction.json"]:
            raise ValueError(f"Invalid ZIP structure: {names}")

    digest = hashlib.sha256(json_path.read_bytes()).hexdigest()
    return json_path, zip_path, digest

if RUN_OFFICIAL_GENERATION and TRAINING_COMPLETE_FOR_OUTPUTS:

    if "model" in globals():
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    model, selected_checkpoint = load_selected_model()
    partial_path = OUTPUT_DIR / "official_generation_progress.json"
    valid_ids = set(official_ids)

    checkpoint_file = Path(selected_checkpoint)
    checkpoint_stat = checkpoint_file.stat()
    progress_signature = {
        "checkpoint": str(checkpoint_file.resolve()),
        "checkpoint_size": int(checkpoint_stat.st_size),
        "checkpoint_mtime_ns": int(checkpoint_stat.st_mtime_ns),
        "tokenizer_sha256": TOKENIZER_SHA256,
        "output_words": int(OUTPUT_WORDS),
        "temperature": float(TEMPERATURE),
        "eval_mem_len": int(EVAL_MEM_LEN),
    }

    def _valid_prediction_map(values):
        result = {}
        if not isinstance(values, dict):
            return result
        for key, value in values.items():
            key = str(key)
            text = value.get("text") if isinstance(value, dict) else value
            if (
                key in valid_ids
                and isinstance(text, str)
                and len(text.split()) == OUTPUT_WORDS
            ):
                result[key] = text
        return result

    in_memory_predictions = globals().get("predictions")
    predictions = {}
    if partial_path.exists():
        try:
            saved_progress = json.loads(partial_path.read_text(encoding="utf-8"))
            if saved_progress.get("signature") == progress_signature:
                predictions.update(_valid_prediction_map(saved_progress.get("predictions", {})))
            else:
                print("Ignoring partial predictions from a different checkpoint/configuration.")
        except Exception as exc:
            print("Could not read the prior partial prediction file; starting from in-memory results:", exc)

    predictions.update(_valid_prediction_map(in_memory_predictions))

    def _save_official_progress():
        payload = {
            "signature": progress_signature,
            "predictions": {
                key: {"text": predictions[key]}
                for key in official_ids
                if key in predictions
            },
        }
        temporary_path = partial_path.with_suffix(".tmp")
        temporary_path.write_text(
            json.dumps(payload, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        temporary_path.replace(partial_path)

    _save_official_progress()
    print(f"Resuming with {len(predictions)}/{len(official_ids)} valid predictions already saved.")

    for item in tqdm(test_examples, desc="Official Phase-1 generation"):
        test_id = str(item["id"])
        if test_id in predictions:
            continue
        try:
            predictions[test_id] = generate_128_word_continuation(
                model,
                item["prompt"],
                temperature=TEMPERATURE,
            )
        except Exception as exc:
            _save_official_progress()
            raise RuntimeError(
                f"Official generation stopped at test ID {test_id}; "
                f"saved {len(predictions)}/{len(official_ids)} predictions to {partial_path}. "
                f"Resume by rerunning this cell after fixing the reported issue."
            ) from exc

        if len(predictions) % 10 == 0:
            _save_official_progress()

    _save_official_progress()
    missing_ids = [key for key in official_ids if key not in predictions]
    if missing_ids:
        raise RuntimeError(f"Generation incomplete; missing {len(missing_ids)} official IDs.")

    prediction_path, zip_path, prediction_sha256 = write_submission(predictions)
    print("prediction.json:", prediction_path)
    print("submission ZIP:", zip_path)
    print("prediction SHA256:", prediction_sha256)

    save_json(OUTPUT_DIR / "submission_manifest.json", {
        "architecture": ARCHITECTURE_NAME,
        "checkpoint": str(selected_checkpoint),
        "num_predictions": len(predictions),
        "first_ids": official_ids[:10],
        "last_ids": official_ids[-10:],
        "prediction_sha256": prediction_sha256,
        "prediction_json": str(prediction_path),
        "submission_zip": str(zip_path),
        "partial_progress": str(partial_path),
        "generation_retry_budgets": list(GENERATION_TOKEN_BUDGETS),
        "training_segment_len": int(SEGMENT_LEN),
        "training_mem_len": int(TRAIN_MEM_LEN),
        "generation_mem_len": int(EVAL_MEM_LEN),
    })

    del model, predictions
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
elif RUN_OFFICIAL_GENERATION:
    print("Official generation skipped: train the model first.")

In [ ]:
manifest = {
    "dataset": DATASET_ID, "config": DATASET_CONFIG, "seed": SEED,
    "architecture": model_config, "architecture_name": ARCHITECTURE_NAME,
    "positional_encoding": "Transformer-XL relative sinusoidal",
    "segment_recurrence": {"enabled": True, "memory_detached": True,
        "training_segment_len": int(SEGMENT_LEN), "training_memory_len": int(TRAIN_MEM_LEN),
        "generation_memory_len": int(EVAL_MEM_LEN)},
    "parameters": int(audit["total_parameters"]),
    "parameter_millions": float(audit["parameter_millions"]),
    "tokenizer": str(TOKENIZER_PATH), "tokenizer_sha256": TOKENIZER_SHA256,
    "vocab_size": VOCAB_SIZE, "train_documents": int(TRAIN_DOCUMENT_COUNT),
    "valid_documents": int(VALID_DOCUMENT_COUNT),
    "document_counts_by_source": stream_split_document_counts,
    "train_tokens": int(len(train_ids)), "valid_tokens": int(len(valid_ids)),
    "prompt_handling": {"max_official_prompt_tokens": int(max(prompt_lengths)),
        "truncation": False,
        "estimated_128_word_completion_tokens": int(CONTEXT_ESTIMATED_COMPLETION_TOKENS)},
    "training": {"epochs": EPOCHS, "steps": MAX_STEPS,
        "tokens_per_optimizer_update": TOKENS_PER_OPTIMIZER_STEP,
        "micro_batch_size": MICRO_BATCH_SIZE, "gradient_accumulation": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
        "gradient_clip": GRAD_CLIP, "evaluation_every_steps": EVAL_EVERY_STEPS,
        "max_validation_batches": MAX_EVAL_BATCHES,
        "training_token_budget_per_epoch": TRAIN_TOKEN_BUDGET,
        "planned_training_token_exposure": TRAIN_TOKEN_BUDGET * EPOCHS,
        "validation_token_budget": VALID_TOKEN_BUDGET},
    "generation": {"recurrent_memory_enabled": True, "output_words": OUTPUT_WORDS,
        "token_budgets": list(GENERATION_TOKEN_BUDGETS),
        "max_bpe_pieces_per_forced_word_boundary": MAX_BPE_TOKENS_PER_WORD,
        "temperature": TEMPERATURE, "top_k": TOP_K, "top_p": TOP_P},
    "official_test_count": len(official_ids), "official_test_ids": official_ids,
    "local_metrics": local_metrics if "local_metrics" in globals() else None,
    "training_summary": training_summary, "output_dir": str(OUTPUT_DIR),
}
save_json(OUTPUT_DIR / "phase1_run_manifest.json", manifest)
print("Saved:", OUTPUT_DIR / "phase1_run_manifest.json")

In [ ]:
import shutil
import os

folder = "/kaggle/working/phase1_transformer_xl_250m_outputs"

zip_path = shutil.make_archive(
    "/kaggle/working/phase1_transformer_xl_250m_outputs",
    "zip",
    root_dir="/kaggle/working",
    base_dir="phase1_transformer_xl_250m_outputs"
)

print("Created:", zip_path)
print("Exists:", os.path.exists(zip_path))
print("Size:", os.path.getsize(zip_path) / (1024**2), "MB")